In [2]:
from google.colab import files
files.upload()

Saving amazon.csv to amazon.csv


{'amazon.csv': b',reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound\r\n1,0mie,5,"Purchased this for my device, it worked as advertised. You can never have too much phone memory, since I download a lot of stuff this was a no brainer for me.",25-10-2023,10,80,35,115,45,1.25,0.139\r\n2,1K3,4,it works as expected. I should have sprung for the higher capacity.  I think its made a bit cheesier than the earlier versions; the paint looks not as clean as before,23-12-2021,21,60,31,91,29,3.8,0.2125\r\n3,1m2,5,"This think has worked out great.Had a diff. bran 64gb card and if went south after 3 months.This one has held up pretty well since I had my S3, now on my Note3.*** update 3/21/14I\'ve had this for a few months and have had ZERO issue\'s since it was transferred from my S3 to my Note3 and into a note2. This card is reliable and solid!Cheers!",21-11-2021,19,25,0,25,25,4.5,0.1365\r\n4,2Cents!,5,It\'

In [3]:
import os
os.listdir()

['.config', 'amazon.csv', 'sample_data']

In [4]:
import pandas as pd

df = pd.read_csv("amazon.csv")
df.head()


,Unnamed: 0,reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound
0,1,0mie,5,"Purchased this for my device, it worked as adv...",25-10-2023,10,80,35,115,45,1.25,0.1390
1,2,1K3,4,it works as expected. I should have sprung for...,23-12-2021,21,60,31,91,29,3.80,0.2125
2,3,1m2,5,This think has worked out great.Had a diff. br...,21-11-2021,19,25,0,25,25,4.50,0.1365
3,4,2Cents!,5,It's mini storage. It doesn't do anything els...,29-04-2023,15,56,47,103,9,2.60,0.1798
4,5,2K1Toaster,5,I have it in my phone and it never skips a bea...,19-10-2022,15,14,14,28,0,0.90,0.1883


In [5]:
import re

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))
    return text.lower()

df['reviewText_cleaned'] = df['reviewText'].apply(clean_text)


In [6]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

df['compound_score'] = df['reviewText_cleaned'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_class'] = df['compound_score'].apply(classify_sentiment)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['reviewText_cleaned'])
y = df['sentiment_class']


In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [10]:
from sklearn.metrics import classification_report, accuracy_score

rf_pred = rf.predict(X_test)
lr_pred = lr.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))


Random Forest Accuracy: 0.9581497797356828
              precision    recall  f1-score   support

    negative       0.93      0.96      0.94       291
     neutral       1.00      0.98      0.99       274
    positive       0.95      0.94      0.94       343

    accuracy                           0.96       908
   macro avg       0.96      0.96      0.96       908
weighted avg       0.96      0.96      0.96       908

